In [11]:
from importlib import import_module

named_libs = [('pandas', 'pd')] # (library_name, shorthand)
for (name, short) in named_libs:
    try:
        lib = import_module(name)
    except:
        print(sys.exc_info())
    else:
        globals()[short] = lib  

libnames = ['requests', 'json', 'pyexasol', 'configparser']
for libname in libnames:
    try:
        lib = import_module(libname)
    except:
        print(sys.exc_info())
    else:
        globals()[libname] = lib
        
### API EXTRACTION

def cleanNsafeExtract():
    try:
        ### Initialize the dataframe
        final_df = pd.DataFrame()
        ### API link
        url = "https://api.hotel-audit.hrs.com/v1/audits/report?secretKey=3AwuZKz9nH&page=1&size=100"
        parsed = json.loads(requests.get(url).text)
        ### Get the number of pages through -> #int(parsed['total_pages'])
        for x in range(int(parsed['total_pages'])):
            url_iter = "https://api.hotel-audit.hrs.com/v1/audits/report?secretKey=3AwuZKz9nH&page="+str(x+1)+"&size=100"
            response = requests.get(url_iter)
            if response.status_code == 403:    
                data = response.text
                parsed = json.loads(data)
                df = pd.DataFrame(parsed.get('results'))
                final_df = final_df.append(df, ignore_index=True)
            else: 
                print("Request failed page: {} ".format(x))
                
        final_df['created_date']= pd.to_datetime(final_df['created_date'])
        final_df['updated_date']= pd.to_datetime(final_df['updated_date'])
        harmonized_df = final_df[final_df.hkey.notnull()]
        #harmonized_df = harmonized_df.loc[:, harmonized_df.columns != 'checked']
    except:
        print('Failed in function cleanNsafe - ')
    return harmonized_df, print(f'Number of records having HOTEL_IDs {final_df.id[final_df.hkey.notnull()].count()}.'), print(f'Number of records with no HOTEL_IDs  {final_df.id[final_df.hkey.isna()].count()}.')    

In [12]:
def cleanNsafeLoad(df): 
    harmonized_df = df
    try:
        #Location of the ini file
        config = configparser.ConfigParser()
        ## Config location CHANGE
        config.read('C:\\Users\\USER\\.spyder-py3\\pfxDET.ini')
        dsn=config['pfxDET']['dsn']
        user=config['pfxDET']['user']
        pwd=config['pfxDET']['pwd']
        schema=config['pfxDET']['schema']
        # Exasol connection
        connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)
        connect.execute("TRUNCATE TABLE DWHPFX.CLEAN_SAFE_HOTELS")
        connect.import_from_pandas(harmonized_df, table = ('DWHPFX','CLEAN_SAFE_HOTELS'))
        stmt = connect.last_statement()
        print(f'Number of records inserted  {stmt.rowcount()}.')
    except:
        print('Failed in function cleanNsafeLoad - check DB connection')
#     return print(f'Number of records inserted  {harmonized_df.id[harmonized_df.hkey.notnull()].count()}.')



In [13]:
df, a, b = cleanNsafeExtract()
cleanNsafeLoad(df)
df.head()

In [79]:
from urllib.parse import urlsplit
parsed = urlsplit("https://api.hotel-audit.hrs.com/v1/audits/report?secretKey=3AwuZKz9nH&page=1&size=100")
print('query  :', parsed.query)

In [80]:
print('query  :', parsed.fragment)

In [6]:
### Initialize the dataframe
final_df = pd.DataFrame()
### API link
url = "https://api.hotel-audit.hrs.com/v1/audits/report?secretKey=3AwuZKz9nH&page=1&size=100"
parsed = json.loads(requests.get(url).text)
### Get the number of pages through -> #int(parsed['total_pages'])
for x in range(int(parsed['total_pages'])):
    url_iter = "https://api.hotel-audit.hrs.com/v1/audits/report?secretKey=3AwuZKz9nH&page="+str(x+1)+"&size=100"
    response = requests.get(url_iter)
    if response.status_code == 403:    
        data = response.text
        parsed = json.loads(data)
        df = pd.DataFrame(parsed.get('results'))
        final_df = final_df.append(df, ignore_index=True)
    else: 
        print("Request failed page: {} ".format(x))

final_df['created_date']= pd.to_datetime(final_df['created_date'])
final_df['updated_date']= pd.to_datetime(final_df['updated_date'])
harmonized_df = final_df[final_df.hkey.notnull()]

In [7]:
harmonized_df.count()

In [8]:
cleanNsafeLoad(df)

In [14]:
config = configparser.ConfigParser()
## Config location CHANGE
config.read('C:\\Users\\USER\\.spyder-py3\\pfxDET.ini')
dsn=config['pfxDET']['dsn']
user=config['pfxDET']['user']
pwd=config['pfxDET']['pwd']
schema=config['pfxDET']['schema']
# Exasol connection
connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)
# df = connect.export_to_pandas("SELECT * FROM DWHBIL.V_LKP_HOTEL LIMIT 100")
# df.head()
connect.import_from_pandas(df, table = ('DWHPFX','CLEAN_SAFE_HOTELS'))
stmt = connect.last_statement()